# Alignment Pipeline Sample Processing Helper

This notebook helps you prepare configurations and execution wrapper scripts on a per-sample basis for the Non-UMI Alignment Pipeline.

### Guidelines:
1. Copy this notebook into your project workspace directory.
2. Modify the configuration parameters in **Step 2** below.
3. Run all cells. This notebook will automatically generate configuration YAMLs and shell script wrappers for all unique samples found in your metadata sheet.
4. We expect that the `patientID` and the `sample type` are joined using an underscore `_` and themselves don't contain underscores. Modify this split logic accordingly if your naming convention differs.

## Step 1: Import Dependencies
Import standard Python libraries needed for file system operations, YAML handling, and data manipulation.

In [ ]:
import os
import glob
import yaml
import pandas as pd
from pathlib import Path

## Step 2: Define Project Paths and Configuration Parameters
Modify the paths below to point to your data, reference genome, metadata sheet, and output directories.

In [ ]:
# ==============================================================================
# 1. Pipeline and Reference Genome Paths
# ==============================================================================
ALIGNMENT_PIPELINE_PATH = "path/to/cloned/directory"  # Absolute path to the cloned pipeline repository
GENOME_FASTA = "/path/to/genome.fa"                   # Reference genome FASTA (must be indexed for BWA)
DBSNP_VCF = "/path/to/dbsnp_all_vcf.gz"               # Known SNP sites VCF (gzipped, required for BQSR)

# ==============================================================================
# 2. Sequencing Target BED Files (Only needed for WES or Panel sequencing)
# ==============================================================================
TARGET_FILE = "Agilent_SureSelectv5.target.bed"       # Absolute path to target regions BED file
BAIT_FILE = "Agilent_SureSelectv5.bait.bed"           # Absolute path to bait regions BED file

# ==============================================================================
# 3. Metadata Sheet Path
# ==============================================================================
METADATA_PATH = "/path/to/metadata_file"              # CSV/TSV describing runs, lanes, reads, fastqs

# ==============================================================================
# 4. Directories for Output Data and Generated Scripts
# ==============================================================================
BASE_OUTPUT_PATH = "/path/to/output/directory"        # Where final Cram/Bam files and QC metrics will go
SCRIPT_PATH = "/path/to/scripts/directory"            # Directory where sample-specific runner scripts will be saved

# ==============================================================================
# 5. Trimming Adapters
# ==============================================================================
ADAPTER_FORWARD_READ = "adapter_forward_read"          # R1 adapter sequence
ADAPTER_REVERSE_READ = "adapter_reverse_read"          # R2 adapter sequence
ADDITIONAL_CUTADPT_PARAMETER = "-u 5"                 # Additional trimming options (e.g. Agilent MBC trimming)

# ==============================================================================
# 6. Sequencing Parameters
# ==============================================================================
SEQ_TYPE = "WES"                                      # Either "WGS", "WES", or "Panel"

# ==============================================================================
# 7. Execution / Environment Settings
# ==============================================================================
CONDA_ACTIVATE_PATH = "~/miniforge3/bin/activate"     # Conda activate script path
RUN_LOCAL = True                                      # True to run locally with --cores, False for SLURM cluster

# ==============================================================================
# 8. SLURM Cluster Specific Profiles
# ==============================================================================
SLURM_SMK_PROFILE = "~/smk-profile-v0.1"              # Profile path (only used if RUN_LOCAL = False)
WORKFLOW_PROFILE = "profile/"                         # Snakemake workflow profile directory

## Step 3: Load and Preview Metadata Sheet
Load the metadata sheet to verify formatting, columns, and rows before starting config generation.

In [ ]:
# Resolve file separator based on file extension
sep = "\t" if METADATA_PATH.split(".")[-1] in ["tsv", "txt"] else ","
df = pd.read_csv(METADATA_PATH, dtype=str, sep=sep)

# Display metadata summary
print(f"Loaded metadata sheet with {df.shape[0]} rows and {df.shape[1]} columns.")
print("Columns present:", list(df.columns))
df.head()

## Step 4: Generate Configuration YAMLs and Runner Scripts
For each unique sample found in the metadata sheet, this step will:
1. Create the output workspace directory.
2. Load the base config `config/config.yaml` template.
3. Generate a custom YAML config containing correct paths and library kits for the sample.
4. Write a script wrapper `.sh` to activate the conda environment and trigger the Snakemake pipeline.

In [ ]:
# Ensure the output folders exist
workdir_base = Path(f"{BASE_OUTPUT_PATH}/view-by-pid/")
bash_folder = Path(f"{SCRIPT_PATH}")

os.makedirs(bash_folder, exist_ok=True)
os.makedirs(workdir_base, exist_ok=True)

# Load the template configuration file
template_config_path = Path("config/config.yaml")
if not template_config_path.exists():
    # Fall back to relative path within pipeline repo if notebook is running from another location
    template_config_path = Path(ALIGNMENT_PIPELINE_PATH) / "config" / "config.yaml"

yaml_dict = yaml.safe_load(template_config_path.read_text())

# Process each unique sample name
count = 0
for sample_name in df.SAMPLE_NAME.unique():
    count += 1
    print(f"[{count}] Processing sample: {sample_name}")
    
    # Extract Patient ID (PID) and Sample Type by splitting on underscore
    pid_part = sample_name.split("_")[0]
    sample_part = sample_name.split("_")[1]
    
    # Define output folder paths for sample
    sample_workdir = workdir_base / pid_part / sample_part / "merged-alignment"
    sample_logdir = sample_workdir / "logs"
    os.makedirs(sample_workdir, exist_ok=True)
    
    # Map paths and inputs into the custom config dict
    yaml_dict["work_dir"] = str(sample_workdir)
    yaml_dict["log_dir"] = str(sample_logdir)
    yaml_dict["pid"] = pid_part
    yaml_dict["sample"] = sample_part
    yaml_dict["genome"] = GENOME_FASTA
    yaml_dict["dbsnp"] = DBSNP_VCF
    yaml_dict["target_regions"] = TARGET_FILE
    yaml_dict["bait_regions"] = BAIT_FILE
    yaml_dict["SeqType"] = SEQ_TYPE
    
    # Extract and verify the library preparation kit
    sample_rows = df[df.SAMPLE_NAME == sample_name]
    lib_kits = sample_rows["LIBRARY_PREP_KIT"].unique()
    if len(lib_kits) > 1:
        raise ValueError(f"Error: Sample {sample_name} has multiple library prep kits: {lib_kits}")
    yaml_dict["library_prep_kit"] = lib_kits[0]
    
    # Adapter trimming setup
    yaml_dict["trim_adapters"] = True   
    yaml_dict['adapter_forward_read'] = [ADAPTER_FORWARD_READ]
    yaml_dict['adapter_reverse_read'] = [ADAPTER_REVERSE_READ]
    yaml_dict['extra_cutadapt_params'] = ADDITIONAL_CUTADPT_PARAMETER
    if ADDITIONAL_CUTADPT_PARAMETER == "":
        yaml_dict.pop('extra_cutadapt_params', None)

    yaml_dict["metadata"] = METADATA_PATH
    
    # Save sample-specific YAML config file
    config_file = sample_workdir / f"{sample_name}.yaml"
    with open(config_file, "w") as ff:
        yaml.dump(yaml_dict, ff)
        
    # Assemble Snakemake execution command
    if RUN_LOCAL:
        # Local execution using --cores instead of SLURM profiles
        cmd = (
            "module purge; source "
            f"{CONDA_ACTIVATE_PATH}"
            " alignment-base && cd "
            + yaml_dict["work_dir"]
            + " "
            + "&& snakemake --cores 8 "
            + f"--workflow-profile {WORKFLOW_PROFILE} "
            + " --use-singularity --use-conda --conda-frontend mamba "
            + f"--configfile {str(config_file)} "
            + f"--snakefile {ALIGNMENT_PIPELINE_PATH}/workflow/Snakefile"
        )
    else:
        # Cluster execution utilizing SLURM profile
        cmd = (
            "module purge; source "
            f"{CONDA_ACTIVATE_PATH}"
            " alignment-base && cd "
            + yaml_dict["work_dir"]
            + " "
            + f"&& snakemake --profile {SLURM_SMK_PROFILE} "
            + f"--workflow-profile {WORKFLOW_PROFILE} "
            + " --use-singularity --use-conda --conda-frontend mamba -j 10 "
            + f"--configfile {str(config_file)} "
            + f"--snakefile {ALIGNMENT_PIPELINE_PATH}/workflow/Snakefile"
        )
        
    # Write execution wrapper script
    runner_script = bash_folder / f"{count}.sh"
    with open(runner_script, "w") as handle:
        handle.write(cmd)
        
print(f"\nDone! Generated {count} configuration YAMLs and run scripts in: {bash_folder}")